In [22]:
#Limpeza da tabela campeonato-brasileiro-full.csv

# Esse script segue uma ordem lógica de limpeza de dados:
#1. Carregar
#2. Tratar valores nulos "disfarçados"
#3. Limpar texto (espaços, maiúsculas/minúsculas)
#4. Corrigir coluna vencedor
#5. Corrigir datas
#6. Corrigir números da coluna arrecadacao e coluna visitante, mandante.
#7. Trata outliers valores absurdos
#8. Remover duplicatas
#9. Salvar




In [23]:
import pandas as pd
import numpy as np

#carregamento de dados brutos e bagunçados
df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-full.csv")

#tratamento de valores nulos
fake_null_values = ["", " ", "N/A", "-", "NULL", "null", "NaN", "nan", "<NA>"]
df = df.replace(fake_null_values, np.nan)


In [24]:
#limpar espaços em branco das colunas
colunas_texto = df.select_dtypes(include="object").columns
for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

#padronizar maisculas e minusculas
coluna_de_sigla = ["mandante_Estado", "visitante_Estado"]
coluna_de_nomes = ["mandante", "visitante", "arena", "tecnico_mandante", "tecnico_visitante"]    
for coluna in coluna_de_nomes:
    df[coluna] = df[coluna].str.title()

for coluna in coluna_de_sigla:
    df[coluna] = df[coluna].str.upper() 

C:\Users\Matheus\AppData\Local\Temp\ipykernel_8512\1674606544.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


In [25]:
#correção da  coluna vencedor (todas as colunas que nao tiver vencedor sera colocado empate)
df["vencedor"] = df["vencedor"].fillna("Empate")
df["vencedor"] = df["vencedor"].str.title()

In [26]:
#correção da coluna de data para formato datetime
#COMENTARIO: A coluna de data possuia diversos fromatos de horario, fazendo com uqe o date time não conseguisse converter todos os valores. A solução desse problema foi criar uma lista de possiveis formatos e converter "manuealmente", até qque todos pudessem virar datetime.
formatos_de_data =  ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%d-%m-%Y","%m/%d/%Y", "%d/%m/%y", "%Y/%m/%d"]
data_convertida = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

for formato in formatos_de_data:
    ainda_sem_data = data_convertida.isna()
    segunda_tentativa = pd.to_datetime(df.loc[ainda_sem_data, "data"], format = formato, errors="coerce")
    data_convertida.loc[ainda_sem_data] = segunda_tentativa
df["data"] = data_convertida
datas_invalidas = df["data"].isna().sum()

In [27]:

#correção as colunas de placar transfromando para numeros inteiros e removendo os possiveis textos na coluna mandante e visitante
for coluna in ["mandante_Placar", "visitante_Placar"]:
    df[coluna] = df[coluna].str.replace(" un","", regex=False)
    df[coluna] = df[coluna].str.replace(",","", regex=False)
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    df[coluna] = df[coluna].astype("Int64")


In [28]:
#correção da coluna de arrecadação (estava em padrão americano e com separador de milhar e foi para o padrão brasileiro)
df["arrecadacao"] = pd.to_numeric(df["arrecadacao"], errors="coerce")

def formatar_arrecadacao(valor):
    if pd.isna(valor):
        return np.nan
    
    texto = f"{valor:,.2f}"
    texto = texto.replace(",", "X")
    texto = texto.replace(".", ",")
    texto = texto.replace("X", ".")
    return f"R$ {texto}"
df["arrecadacao"] = df["arrecadacao"].apply(formatar_arrecadacao)

In [29]:
#trata outliers valores absurdos
limite_maximo_de_gols = 12
for coluna in ["mandante_Placar", "visitante_Placar"]:
    outliers = df[coluna] > limite_maximo_de_gols
    df.loc[outliers, coluna] = np.nan 

duplicatas = df.duplicated(subset="ID").sum()
df = df.drop_duplicates(subset = "ID", keep = "first")  

In [30]:
# DECISÃO DE LIMPEZA: as linhas que ficaram com <NA> foram mantidas, pois nao há informação para saber se houve gol ou não. 
# Preencher com 0 poderia gerar uma informação falsa e remover a linha traria danos a interepretação dos dados. 
# Os dados serão importantes para analises futuras.


#ordenação da tabela de acordo com a ordem das colunas e ordenação por data
ordem_das_colunas = [
    "ID", "rodata", "data", "hora","mandante", "visitante","mandante_Placar", "visitante_Placar", "vencedor","mandante_Estado", "visitante_Estado", "arena","formacao_mandante", "formacao_visitante", "tecnico_mandante", "tecnico_visitante", "arrecadacao",]
df = df[ordem_das_colunas]
df = df.sort_values(by=["data"], ascending=True).reset_index(drop=True)



print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())



Linhas finais: 9165

Valores nulos restantes por coluna:
ID                       0
rodata                   0
data                     0
hora                     0
mandante                 0
visitante                0
mandante_Placar         97
visitante_Placar         0
vencedor                 0
mandante_Estado          0
visitante_Estado         0
arena                    0
formacao_mandante     5140
formacao_visitante    5026
tecnico_mandante      4821
tecnico_visitante     4770
arrecadacao           8808
dtype: int64


In [31]:
#salvamento do arqqquivo levando para a pasta processed, que contem os dados limpos.
print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())


caminho_saida = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-jogos-limpo.csv"
df.to_csv(caminho_saida, index=False)

display(df)


Linhas finais: 9165

Valores nulos restantes por coluna:
ID                       0
rodata                   0
data                     0
hora                     0
mandante                 0
visitante                0
mandante_Placar         97
visitante_Placar         0
vencedor                 0
mandante_Estado          0
visitante_Estado         0
arena                    0
formacao_mandante     5140
formacao_visitante    5026
tecnico_mandante      4821
tecnico_visitante     4770
arrecadacao           8808
dtype: int64


,ID,rodata,data,hora,mandante,visitante,mandante_Placar,visitante_Placar,vencedor,mandante_Estado,visitante_Estado,arena,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,arrecadacao
0,1,1,2003-03-29,16:00,Guarani,Vasco,4,2,Guarani,SP,RJ,Brinco De Ouro,NaN,NaN,NaN,NaN,NaN
1,2,1,2003-03-29,16:00,Athletico-Pr,Gremio,2,0,Athletico-Pr,PR,RS,Arena Da Baixada,NaN,NaN,NaN,NaN,NaN
2,11,1,2003-03-30,18:00,Santos,Parana,2,2,Empate,SP,PR,Vila Belmiro,NaN,NaN,NaN,NaN,NaN
3,9,1,2003-03-30,16:00,Cruzeiro,Sao Caetano,2,2,Empate,MG,SP,Mineirão,NaN,NaN,NaN,NaN,NaN
4,7,1,2003-03-30,16:00,Juventude,Sao Paulo,2,2,Empate,RS,SP,Alfredo Jaconi,NaN,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9160,8906,13,2025-12-07,16:30,Flamengo,Sao Paulo,<NA>,0,Flamengo,RJ,SP,"Maracanã, Rio De Janeiro",4-2-3-1,5-3-2,F. Luís,H. Crespo,"R$ 4.693.377,50"
9161,9159,38,2025-12-07,16:04,Corinthians,Juventude,1,1,Empate,SP,RS,"Neo Química Arena, São Paulo",3-4-3,4-2-3-1,D. Junior,M. Barbieri,"R$ 1.888.899,00"
9162,9162,38,2025-12-07,16:02,Vitoria,Sao Paulo,1,0,Vitoria,BA,SP,"Barradão, Salvador",3-4-2-1,3-5-2,J. Ventura,H. Crespo,NaN
9163,9163,38,2025-12-07,16:04,Botafogo-Rj,Fortaleza,<NA>,2,Botafogo-Rj,RJ,CE,"Estádio Nilton Santos, Rio De Janeiro",4-2-3-1,4-4-2,Davide Ancelotti,Martin Palermo,"R$ 905.640,00"
